# 1. Data Acquisition

Download the 2013–2024 SEC quarterly financial-statement datasets and three FRED series.
Raw files total several GB and are excluded from the repository. The processed-data route starts at Notebook 03.

**Verification:** statically checked; not rerun in this release because raw data are not included. No SEC files were downloaded during validation.
See [data instructions](../data/README.md) for the required `SEC_USER_AGENT` environment variable.

In [ ]:
from pathlib import Path
import sys

# Works from the project root or any folder beneath it.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/analysis_helpers.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project folder.")
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

In [ ]:
import os
import time
import urllib.request
import zipfile

user_agent = os.environ.get("SEC_USER_AGENT", "").strip()
if not user_agent or "@" not in user_agent or "example.com" in user_agent.lower():
    raise RuntimeError("Set SEC_USER_AGENT to a descriptive project name and your contact email in your environment. Do not store it in the repository.")
headers = {"User-Agent": user_agent}
raw = ROOT / "data/raw"
raw.mkdir(parents=True, exist_ok=True)

def download(url, destination, is_zip=False):
    if destination.exists():
        if not is_zip or zipfile.is_zipfile(destination):
            print("Already present:", destination.name)
            return
        raise ValueError("Existing file is not a valid ZIP: " + destination.name)
    partial = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(request, timeout=120) as response, partial.open("wb") as output:
                while chunk := response.read(1 << 20):
                    output.write(chunk)
            if is_zip and not zipfile.is_zipfile(partial):
                raise ValueError("Download is not a ZIP")
            partial.replace(destination)
            print("Saved:", destination.name)
            return
        except (OSError, ValueError):
            if attempt == 2:
                raise RuntimeError("Download failed: " + destination.name) from None
            time.sleep(5)

for year in range(2013, 2025):
    for quarter in range(1, 5):
        name = f"{year}q{quarter}.zip"
        download("https://www.sec.gov/files/dera/data/financial-statement-data-sets/" + name,
                 raw / name, is_zip=True)
        time.sleep(0.3)
for series in ("FEDFUNDS", "CPIAUCSL", "INDPRO"):
    download("https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series,
             raw / f"fred_{series}.csv")

## Download behavior

Existing ZIPs are preserved. New downloads use a temporary `.part` file, retry up to three times, and fail explicitly if unsuccessful. The contact header is never printed or saved in notebook output.

Sources: [SEC datasets](https://www.sec.gov/data-research/sec-markets-data/financial-statement-data-sets), [FRED](https://fred.stlouisfed.org/).